# Agricultural Field Carbon Credit Estimation using coupling of Satellite Embeddings and RAG

## Specification-Driven Development & Deployment (SDD)

**Specification:** `AGRICULTURAL-FIELD-CARBON-CREDIT-SDD-1.0`  
**Status:** Executable deployment specification / prototype acceptance run  
**Audience:** Engineering, GIS, agronomy, MLOps, and review teams

This notebook links requirements to data contracts, model configuration, implementation, acceptance checks, and deployment operations. Run top-to-bottom from the project root using the project virtual environment. The live Sentinel-2 step accesses the public Microsoft Planetary Computer STAC API and may take several minutes.

**Capability boundary:** this is not a verified carbon-credit estimate or eligibility score. Current inference uses Sentinel-2 spectral indices and a provisional dNBR rule; learned satellite embeddings and RAG retrieval are a future extension and are not currently used.

## 1. Product specification and decision boundary

The system monitors `input_farmboundary/` for a new GeoJSON polygon, ESRI Shapefile, or CSV bounding box. Once stable, it validates and normalizes the field to WGS84, fetches Sentinel-2 L2A scenes for five complete years, screens seasonal NBR change, writes a JSON report, and records request state and model metrics.

**Non-goals / safety constraints:** this prototype does not establish crop type, prove stubble burning or its absence, estimate avoided emissions, or issue carbon credits. A dNBR threshold is a review signal only. Production credit decisions need an approved methodology, verified practice records, ground truth, uncertainty treatment, and independent review. No LLM, RAG component, or legacy classifier participates in measurement or eligibility decisions.

**Input privacy:** boundary geometry can identify a farmer or parcel. Keep real boundary files and reports in access-controlled storage; do not publish them or commit them to source control.

## 2. Testable requirements and traceability

| ID | Requirement | Acceptance evidence |
|---|---|---|
| SDD-01 | Accept GeoJSON, Shapefile, and bbox CSV; normalize projected coordinates using the declared CRS. | Synthetic format examples parse; each is about 2 ha; projected-CRS and invalid-input unit tests pass. |
| SDD-02 | Process multiple fields in one file with unique field IDs. | Per-field report paths are produced; duplicate IDs fail validation. |
| SDD-03 | Poll the inbox hourly, wait for file-copy completion, record failures, and avoid duplicate successful processing. | CLI defaults to 3600 seconds; watcher tests exercise idempotency and report creation. |
| SDD-04 | Obtain Sentinel-2 L2A data for five complete calendar years; screen seasonal NBR change with SCL masks. | Live report contains years, scene counts, NDVI/NBR/dNBR and model version. |
| SDD-05 | Explicitly distinguish a screening signal, no detected signal, and unavailable evidence. | Report status values and schema; unit tests cover missing-year behavior. |
| SDD-06 | Do not manufacture crop classifications or credit/emissions claims. | Report schema requires `not_assessed` and null credit fields. |
| SDD-07 | Persist traceable reports and experiment metadata. | JSON schema validates the report; local MLflow SQLite contains the run and per-year dNBR metrics. |

A passing prototype acceptance run is not scientific validation. Threshold calibration and carbon-credit eligibility remain blocked pending locally collected ground truth and methodology review.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "main.py").exists() or not (ROOT / "src" / "pipeline.py").exists():
    raise RuntimeError("Start Jupyter with the repository root as the working directory.")

DATA_DIR = ROOT / "data"
MODEL_DIR = ROOT / "models"
INBOX_DIR = ROOT / "input_farmboundary"
REPORT_DIR = ROOT / "outputs" / "farm_boundary_reports"
BOUNDARY_SCHEMA_PATH = DATA_DIR / "schemas" / "boundary_request.schema.json"
REPORT_SCHEMA_PATH = DATA_DIR / "schemas" / "screening_report.schema.json"
MODEL_CONFIG_PATH = MODEL_DIR / "sentinel2_screening_v1.json"

print({"project_root": str(ROOT), "python": sys.executable, "python_version": sys.version.split()[0]})

## 3. Data and model readiness gate

`data/dataset_registry.json` records the provenance and limitations of every current and legacy dataset. `models/model_registry.json` explicitly has no production crop classifier. The old `fao_crop_reference.csv`, `satellite_embeddings.npy`, and `crop_classifier.joblib` are retained for audit only; they are generated US-crop demo artifacts and are prohibited from the Haryana inference path.

`models/sentinel2_screening_v1.json` is the active, versioned deterministic screening configuration loaded by the implementation. Verify the implementation and this file agree before a release.

In [ ]:
from jsonschema import Draft202012Validator

dataset_registry = json.loads((DATA_DIR / "dataset_registry.json").read_text(encoding="utf-8"))
model_registry = json.loads((MODEL_DIR / "model_registry.json").read_text(encoding="utf-8"))
model_config = json.loads(MODEL_CONFIG_PATH.read_text(encoding="utf-8"))
boundary_schema = json.loads(BOUNDARY_SCHEMA_PATH.read_text(encoding="utf-8"))
report_schema = json.loads(REPORT_SCHEMA_PATH.read_text(encoding="utf-8"))

Draft202012Validator.check_schema(boundary_schema)
Draft202012Validator.check_schema(report_schema)
assert model_registry["production_crop_classifier"] is None
assert model_registry["production_burn_scar_model"]["model_id"] == model_config["model_id"]
assert model_registry["production_burn_scar_model"]["version"] == model_config["version"]
assert model_config["project_name"] == "Agricultural Field Carbon Credit Estimation using coupling of Satellite Embeddings and RAG"
assert model_config["analysis_years"] == 5
assert model_config["dnbr_threshold"] > 0  # provisional screen, not a validated cutoff
assert all(entry["use_in_current_pipeline"] is False for entry in dataset_registry["legacy_datasets"])

print("Data contracts valid; production crop model correctly remains unregistered.")
print({key: model_config[key] for key in ("project_name", "model_id", "version", "method", "dnbr_threshold", "analysis_years")})

## 4. Deployed component chain

1. **Scheduler / API:** `main.py --watch` scans the top-level inbox hourly; FastAPI can enqueue GeoJSON requests.
2. **Boundary contract:** parser validates geometry, IDs, CRS, and geodesic area; multi-feature files yield individual field reports.
3. **Orchestration:** LangGraph runs validation → Sentinel-2 analysis → report persistence.
4. **Earth observation:** STAC search selects up to three low-cloud scenes per configured seasonal window; L2A B04/B08/B12 and SCL are processed as COGs.
5. **Screening rule:** median NBR and pre-minus-post dNBR produce `possible_burn_scar_signal`, `burn_scar_not_detected`, or `indeterminate`.
6. **Evidence trail:** per-field JSON goes to `outputs/farm_boundary_reports/`; SQLite request queue stores status/errors; MLflow SQLite stores config identity, dNBR metrics, and report artifact.

The notebook runs the same `run_request` entry point; it does not maintain a separate inference implementation. Embedding generation and RAG are documented as future components, not represented as current functionality.

In [ ]:
from src.example_boundaries import create_examples
from src.boundaries import parse_boundary_file

example_files = create_examples(INBOX_DIR)
boundary_examples = [path for path in example_files if path.suffix.lower() in {".geojson", ".csv", ".shp"}]
parsed_examples = [parse_boundary_file(path)[0] for path in boundary_examples]
example_summary = [
    {"format": field.source_file.suffix.lower(), "field_id": field.field_id, "area_ha": round(field.area_ha, 4)}
    for field in parsed_examples
]
assert len(parsed_examples) == 3
assert all(field.field_id == "synthetic_haryana_2ha" for field in parsed_examples)
assert all(1.95 <= field.area_ha <= 2.05 for field in parsed_examples)
print(example_summary)
print("Examples are under input_farmboundary/examples and are not scanned by the top-level inbox watcher.")

## 5. Offline acceptance tests

These test boundary formats, projected-CRS conversion, invalid input handling, LangGraph report construction, watcher idempotency, and conservative missing-data decisions without downloading satellite imagery. They are mandatory before deployment.

In [ ]:
test_result = subprocess.run(
    [sys.executable, "-m", "pytest", str(ROOT / "tests"), "-q"],
    cwd=ROOT,
    text=True,
    capture_output=True,
)
print(test_result.stdout)
if test_result.returncode != 0:
    print(test_result.stderr)
    raise RuntimeError("Offline acceptance tests failed; stop deployment.")
print("Offline acceptance tests passed.")

## 6. End-to-end live demonstration

Set `RUN_LIVE_SATELLITE` to `False` to review the notebook without network access. With it enabled, the next cell sends only the synthetic demonstration boundary to the public STAC catalog. It processes the configured five complete years, saves a real pipeline report and MLflow run under the project `outputs/`, and validates the report against the checked-in JSON schema. This is pipeline integration evidence only—not field truth or detector validation.

In [ ]:
RUN_LIVE_SATELLITE = True
live_report_path = None
if RUN_LIVE_SATELLITE:
    from src.pipeline import run_request

    demo_boundary_path = INBOX_DIR / "examples" / "synthetic_haryana_2ha.geojson"
    run_state = run_request(demo_boundary_path, ROOT)
    live_report_path = Path(run_state["report_path"])
    live_report = json.loads(live_report_path.read_text(encoding="utf-8"))
    Draft202012Validator(report_schema).validate(live_report)
    yearly = live_report["analysis"]["years"]
    assert len(yearly) == model_config["analysis_years"]
    assert live_report["analysis"]["crop_classification"]["status"] == "not_assessed"
    assert live_report["analysis"]["carbon_credit"]["eligible"] is None
    assert live_report["analysis"]["carbon_credit"]["estimated_emissions_avoided_tco2e"] is None
    print("Report:", live_report_path)
    print("Screening model:", live_report["analysis"]["screening_model"])
    print("Yearly results:", [(row["year"], row["status"], row["dnbr"]) for row in yearly])
    print("Comparison:", live_report["analysis"]["comparison"]["trend"])
else:
    print("Live satellite integration skipped. Set RUN_LIVE_SATELLITE=True to run it.")

## 7. MLflow evidence and release gate

A deployable run must have one schema-valid report, five complete-year entries, the exact model ID/version and threshold, and a local MLflow record. The watcher records retryable failures; do not turn missing or invalid observations into a no-burn result. A model-config change is a versioned release and requires the full offline tests plus a reviewed live sample.

In [ ]:
if RUN_LIVE_SATELLITE:
    import mlflow

    tracking_db = ROOT / "outputs" / "mlflow.db"
    mlflow.set_tracking_uri(f"sqlite:///{tracking_db.as_posix()}")
    experiment = mlflow.get_experiment_by_name("agricultural-field-carbon-credit-estimation-rag")
    assert experiment is not None, "MLflow experiment is missing."
    runs = mlflow.search_runs([experiment.experiment_id], max_results=100)
    assert not runs.empty, "No MLflow run was recorded."
    assert f"metrics.dnbr_{yearly[0]['year']}" in runs.columns
    print({"tracking_db": str(tracking_db), "experiment": experiment.name, "runs": len(runs)})
else:
    print("MLflow gate is deferred until the live run is enabled.")

## 8. Deployment procedure (Windows host)

1. Provision a dedicated service account, private project directory, and Python environment; install the checked-in `requirements.txt` and run `python -m pip check`.
2. Restrict ACLs on `input_farmboundary/`, `outputs/`, MLflow SQLite, and logs to the service account and authorized reviewers. Keep real field geometry out of Git and notebook outputs.
3. Run the offline acceptance suite. Review one live, synthetic end-to-end result and verify the schema, model config version, and MLflow artifact.
4. Install the hourly watcher as an **operator-managed** Task Scheduler job or Windows service. Do not run a second competing watcher against the same SQLite queue. Set the action to `<venv>\\Scripts\\python.exe <repository>\\main.py --watch` and the working directory to the repository root (for example, `D:\\FarmCarbonCredit_RAG`); run under the service account, at startup, with restart-on-failure and operational logging. Registration requires local admin/operator approval and is intentionally not performed by this notebook.
5. If the API is required, start `python main.py --serve` on loopback behind an authenticated TLS reverse proxy. Do not expose unauthenticated FastAPI directly to a public network. Apply request-size limits, rate limits, and access logging at the proxy.
6. Observe `GET /health`, `GET /requests`, the request queue, report creation, MLflow runs, disk usage, and STAC/network failures. Alert on repeated failures and stale processing records.

Native Windows excludes vLLM. Deploy optional vLLM only in a separately provisioned Linux/WSL or Linux host; it is not on the screening or carbon decision path.

## 9. Rollback, incident handling, and change control

- Stop/disable the watcher or API service before changing runtime configuration or restoring artifacts.
- Preserve the failing input, queue error, report, model config/version, Python lock/environment details, and logs under access control; do not silently mark it processed.
- Roll back code and the versioned `models/sentinel2_screening_v*.json` configuration together. Never overwrite a model version that produced prior reports.
- Keep SQLite backups consistent: stop writers, then back up the request queue and MLflow database together with the associated reports.
- Re-run offline tests, schema validation, and a synthetic live smoke test before re-enabling the watcher.
- For suspected false positives or negatives, label an independent field-review case; do not alter farmer/carbon status based on this prototype output.

## 10. Final acceptance summary

The notebook is complete when the data/model readiness gate passes, all offline tests pass, the example boundary formats are each approximately 2 ha, the live report validates with five complete years, crop and carbon outputs remain unassessed, and the MLflow record is present. Production scientific validation and carbon-credit authorization are explicit external gates—not implied by a successful deployment.